#### 1. Install dependencies

In [63]:
%pip install --quiet google-adk requests

#### 2. Imports/configuration

Enter your Google Maps Geocoding API key after running cell below. Re-enter it after the runtime restarts.

In [45]:
import getpass
from typing import Dict, Any

import os

import requests

# --- Configuration ---
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = "qwiklabs-gcp-01-06373caf63ce"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

# Geocoding API key is masked in the UI and not stored.
GOOGLE_MAPS_API_KEY = getpass.getpass("Geocoding API key: ")

# The NWS API requires a User-Agent header.
NWS_USER_AGENT = "challenge-1-weather-agent-colab (student-02-730f46eb80e1@qwiklabs.net)"

Geocoding API key: ··········


#### 3. Tool: Get weather from the National Weather Service API

Takes latitude and longitude and returns the current forecast.

In [46]:
def get_weather(latitude: float, longitude: float) -> Dict[str, Any]:
    """Retrieve the current weather forecast for a location.

    Use the U.S. National Weather Service (NWS) API. The NWS API
    resolves the latitude/longitude to a forecast grid endpoint, then
    fetches the forecast periods from that endpoint.

    Args:
        latitude: The latitude of the location in decimal degrees.
        longitude: The longitude of the location in decimal degrees.

    Returns:
        A dictionary containing the forecast. On success it has the keys:
            ``status`` (str): "success".
            ``period`` (str): The name of the forecast period (e.g. "Tonight").
            ``temperature`` (str): The temperature and unit (e.g. "72 F").
            ``forecast`` (str): A short human-readable forecast.
            ``detailed_forecast`` (str): A longer forecast description.
        On failure it returns a dictionary with keys ``status`` ("error")
        and ``error_message`` (str).
    """
    headers = {"User-Agent": NWS_USER_AGENT, "Accept": "application/geo+json"}

    try:
        # Step 1: Resolve the point to a forecast grid endpoint.
        points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
        points_resp = requests.get(points_url, headers=headers, timeout=10)
        points_resp.raise_for_status()
        forecast_url = points_resp.json()["properties"]["forecast"]

        # Step 2: Fetch the forecast from the resolved endpoint.
        forecast_resp = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_resp.raise_for_status()
        periods = forecast_resp.json()["properties"]["periods"]

        if not periods:
            return {
                "status": "error",
                "error_message": "No forecast periods were returned for this location.",
            }

        current = periods[0]
        return {
            "status": "success",
            "period": current["name"],
            "temperature": f"{current['temperature']} {current['temperatureUnit']}",
            "forecast": current["shortForecast"],
            "detailed_forecast": current["detailedForecast"],
        }
    except requests.exceptions.RequestException as exc:
        return {
            "status": "error",
            "error_message": (
                f"Failed to retrieve weather data: {exc}."
            ),
        }
    except (KeyError, IndexError) as exc:
        return {
            "status": "error",
            "error_message": f"Unexpected response format from NWS API: {exc}.",
        }

#### 4. Tool: Geocode a place using the Google Maps Geocoding API

Converts city/state into latitude and longitude.

In [103]:
def geocode_place(place: str) -> Dict[str, Any]:
    """Convert a city/state into latitude and longitude coordinates.

    Uses the Google Maps Geocoding API to resolve a place
    description (i.e. ``"Austin, TX"`` or ``"Seattle, Washington"``)
    into coordinates.

    Args:
        place: A free-form place description such as a city and state.

    Returns:
        A dictionary containing the geocoding result. On success it has
        the keys:
            ``status`` (str): "success".
            ``latitude`` (float): The latitude in decimal degrees.
            ``longitude`` (float): The longitude in decimal degrees.
            ``formatted_address`` (str): The normalized address string.
        On failure it returns a dictionary with keys ``status`` ("error")
        and ``error_message`` (str).
    """
    endpoint = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": place, "key": GOOGLE_MAPS_API_KEY}

    try:
        resp = requests.get(endpoint, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()

        if data.get("status") != "OK" or not data.get("results"):
            return {
                "status": "error",
                "error_message": (
                    f"Could not geocode '{place}'. API status: "
                    f"{data.get('status', 'UNKNOWN')}."
                ),
            }

        result = data["results"][0]
        location = result["geometry"]["location"]
        return {
            "status": "success",
            "latitude": location["lat"],
            "longitude": location["lng"],
            "formatted_address": result["formatted_address"],
        }
    except requests.exceptions.RequestException as exc:
        return {
            "status": "error",
            "error_message": f"Failed to reach the Geocoding API: {exc}.",
        }

#### 5. Weather sub-agent

Gathers U.S. weather via `geocode_place` and `get_weather` functions as tools.

In [104]:
from google.adk.agents import Agent

weather_agent = Agent(
    name="weather_agent",
    model="gemini-2.5-flash",
    description=(
        "An agent that answers questions about the current weather for any "
        "U.S. city or state."
    ),
    instruction=(
        "You are a helpful weather assistant. When a user asks about the "
        "weather in a place (such as a city and state), follow these steps:\n"
        "1. Call the `geocode_place` tool with the place the user mentioned "
        "to obtain its latitude and longitude.\n"
        "2. Call the `get_weather` tool with that latitude and longitude to "
        "retrieve the forecast.\n"
        "3. Report the weather to the user in a friendly, concise sentence, "
        "including the temperature and a short description.\n\n"
        "If either tool returns an error status, apologize and explain the "
        "problem clearly. Remember that the National Weather Service only "
        "covers locations within the United States, so if a user asks about "
        "a place outside the U.S., let them know you can only provide U.S. "
        "weather. If the user does not specify a state, ask for clarification "
        "before geocoding."
        "4. Infer the state if you can deduce fairly accurately. \n"
        "5. If the user provides a city name that's also a state name, "
        "assume it's a city and deduce which state it's in if possible."
    ),
    tools=[geocode_place, get_weather],
)

#### 6. Search sub-agent

Answers general questions using `google_search`.

In [105]:
from google.adk.agents import Agent
from google.adk.tools import google_search

search_agent = Agent(
    name="search_agent",
    model="gemini-2.5-flash",
    description=(
        "An agent that answers general knowledge questions and looks up "
        "current information from the web using Google Search."
    ),
    instruction=(
        "You are a helpful research assistant. Use the `google_search` tool "
        "to find accurate, up-to-date information for the user's question. "
        "Summarize the findings in a clear, concise answer and cite the key "
        "facts you found. If you cannot find a good answer, say so honestly."
    ),
    tools=[google_search],
)

#### 7. Root agent

The entry point for all user requests. Wraps two sub-agents as `AgentTool`s and delegates to the appropriate one.


In [106]:
from google.adk.agents import Agent
from google.adk.tools.agent_tool import AgentTool

root_agent = Agent(
    name="root_agent",
    model="gemini-2.5-flash",
    description=(
        "A routing agent that delegates user requests to specialized "
        "sub-agents for weather and web search."
    ),
    instruction=(
        "You are a router/orchestrator. You do not answer questions "
        "directly yourself. Instead, examine each user request and delegate "
        "it to the correct specialized tool:\n"
        "1. If the request is about the weather, forecast, temperature, or "
        "climate conditions in a location, call the `weather_agent` tool "
        "with the user's request.\n"
        "2. For any other question — general knowledge, current events, "
        "facts, lookups, or anything requiring web information — call the "
        "`search_agent` tool with the user's request.\n"
        "After the chosen sub-agent responds, relay its answer back to the "
        "user clearly and concisely. If a request contains both a weather "
        "part and a search part, call both tools and combine their answers."
    ),
    tools=[
        AgentTool(agent=weather_agent),
        AgentTool(agent=search_agent),
    ],
)

#### Run the root agent

Start the conversation with the root agent.

In [107]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP_NAME = "sub_agents_app"
USER_ID = "user_1"
SESSION_ID = "session_1"

session_service = InMemorySessionService()

runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
)


async def ask_agent(query: str) -> None:
    """Send a query to the root agent and print the final response.

    Args:
        query: The natural-language question from the user.
    """
    # Ensure a session exists.
    session = await session_service.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
    )
    if session is None:
        await session_service.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
        )

    content = types.Content(role="user", parts=[types.Part(text=query)])

    print(f"\nUser: {query}")
    async for event in runner.run_async(
        user_id=USER_ID, session_id=SESSION_ID, new_message=content
    ):
        # Show which sub-agent tool the root agent delegated to.
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.function_call:
                    print(f"  [ROUTE] Delegating to: {part.function_call.name}")
        if event.is_final_response() and event.content and event.content.parts:
            print("Agent:", event.content.parts[0].text)

#### 9. Interact with the root agent

Weather queries route to the weather agent; general queries route to the search agent.

In [108]:
await ask_agent("What's the weather in Jamaica?")


User: What's the weather in Jamaica?
  [ROUTE] Delegating to: weather_agent
Agent: I can only provide weather information for locations within the United States. Jamaica is a country outside the U.S.


In [109]:
await ask_agent("What about Seattle, Washington?")


User: What about Seattle, Washington?
  [ROUTE] Delegating to: weather_agent
Agent: The weather in Seattle, Washington this afternoon is areas of smoke with a high temperature of 87 F.


In [110]:
await ask_agent("Best pizza place in NYC?")


User: Best pizza place in NYC?
  [ROUTE] Delegating to: search_agent
Agent: Defining the "best" pizza place in New York City is a subjective endeavor, as the city boasts an incredible diversity of styles and acclaimed pizzerias. However, several establishments consistently receive high praise from critics and locals alike, offering everything from classic New York slices to Neapolitan and Detroit-style pies. Recent recommendations from 2025 and 2026 highlight a few standouts.

**Highly Recommended Pizzerias in NYC:**

*   **L'Industrie Pizzeria** is frequently cited, with a strong fanbase on platforms like Reddit. Known for its crust and creative toppings like basil-topped burrata and bacon fig jam pie, L'Industrie has even been dubbed the best street pizza in the world and has received several national rankings. Owner Massimo Laveglia, originally from Tuscany, incorporates Italian pizza-making techniques, including a three-day cold fermentation for a light and crispy dough.
*   **Una

In [111]:
await ask_agent("Very briefly explain how clouds are formed")


User: Very briefly explain how clouds are formed
  [ROUTE] Delegating to: search_agent
Agent: Clouds are formed when invisible water vapor in the air condenses into visible water droplets or ice crystals. This process begins with water evaporating from the Earth's surface, turning into water vapor as it heats up. As this warm, moist air rises into the atmosphere, it cools and expands due to lower atmospheric pressure. As the air cools, it can no longer hold all of its water vapor, reaching a saturation point. The water vapor then condenses around tiny airborne particles, such as dust, salt crystals, or pollen, known as condensation nuclei. A large accumulation of these tiny water droplets or ice crystals then becomes visible as a cloud.
